# SafeCrop · Step 1: make the noisy test images (Google Colab version)

This notebook takes your **clean leaf photos** (about 20 to 30 per disease) and makes **three noisier copies** of each one:

| Level | Meaning |
|---|---|
| **L0** | the clean photo (resized only) |
| **L1** | mild noise: a person should still recognize the disease easily |
| **L2** | medium noise: borderline, people may hesitate |
| **L3** | severe noise: hard or impossible to recognize |

It saves every image as a PNG plus a spreadsheet, `metadata.csv`, that the inference notebook will read next. It runs on Colab's free CPU runtime (no GPU needed) in a minute or two.

**Before you run it:** have your clean photos as one or more **zip files**, each with a `clean images` folder and one subfolder per disease:

```
clean.zip
  clean images/
    Rice stem borer(info rich)/   Dead_heart (1).jpg ...
    Potato late blight/
    ...
```

The top folder can also be called `clean` or `clean_images`, and the zip can have any name. Disease folders don't have to match exactly: `Rice stem borer`, `Rice stem borer(info rich)` or `Potato___Late_blight` are all recognised. If you have **several zips**, the notebook reads all of them; if one disease appears in two zips, its photos are combined (exact duplicate files are used once).

**How to run:** in the next section choose how to give Colab your zips (Google Drive, or upload from your computer), then click **Runtime → Run all**.

## How the noise is made

We use five kinds of damage that phone photos really suffer in the field. Four come from the **`imagecorruptions`** library, which reproduces the standard robustness benchmark by Hendrycks & Dietterich (ICLR 2019). That benchmark has severities 1 to 5; we use **1, 3 and 5** as our L1, L2 and L3. The fifth, *low light*, is our own function, because the benchmark only has a "too bright" effect.

| Noise type | What it imitates | How it's made | L1 / L2 / L3 setting |
|---|---|---|---|
| Sensor noise | grainy photo from a cheap camera | `gaussian_noise`: random speckles added to every pixel | noise strength 0.08 / 0.18 / 0.38 |
| Hand-shake blur | the phone moved during the shot | `motion_blur`: smears the image in a random direction | blur length 10 / 15 / 20 px |
| Out of focus | the camera focused on the background | `defocus_blur`: blurs with a round lens-shaped kernel | blur radius 3 / 6 / 10 px |
| Low light | evening or deep shade; the phone brightens a dark shot | our `low_light`: less light, more grain, faded colours | light reaching the camera 30% / 12% / 5% |
| Compression | photo sent over Messenger or WhatsApp | `jpeg_compression`: saves at low JPEG quality | quality 25 / 15 / 7 (out of 100) |

**One noise type per photo.** Each photo gets one noise type at all three levels, so it has 4 versions (L0 to L3). Types are handed out in turn, so each disease ends up with 4 photos per type. With 8 diseases × 20 photos that is **160 clean + 480 noisy = 640 images**, and every noisy image has a clean partner to compare with.

In [ ]:
# Install the noise library (about 10 seconds). Safe to run again.
import subprocess, sys
_ = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecorruptions"], check=True)

In [ ]:
import os, sys, zlib, random, shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from imagecorruptions import corrupt

print("numpy", np.__version__, "| pandas", pd.__version__)
# Note: three of the library's effects (glass_blur, fog, gaussian_blur) crash on NumPy 2.
# We don't use them, so this is fine.

## Get your clean photos into Colab

Pick **one** way by setting `SOURCE` in the cell below:

* **`"drive"` (recommended):** put your zip files in a Google Drive folder called **`SafeCrop`** (in *My Drive*). Colab asks for permission to open your Drive; click *Connect to Google Drive* and allow it. The zips stay in Drive, so you don't upload them again next time, and the results are saved back into `SafeCrop/output`.
* **`"upload"`:** a *Choose files* button appears under the cell; pick your zip file(s). Uploaded files are deleted when Colab disconnects, so you'll upload again each session. Download the results at the end.

Either way, the zips are unpacked into `/content/data`.

In [ ]:
SOURCE = "drive"                 # "drive" or "upload"
DRIVE_FOLDER = "/content/drive/MyDrive/SafeCrop"   # the Drive folder that holds your zip files
DATA_DIR = "/content/data"       # where the zips are unpacked

import zipfile, glob

def unpack(zip_path):
    # unzip into its own folder, skipping the __MACOSX junk that Mac zips include
    dest = os.path.join(DATA_DIR, os.path.splitext(os.path.basename(zip_path))[0])
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist()
                   if not m.startswith("__MACOSX") and not os.path.basename(m).startswith("._")]
        z.extractall(dest, members)
    print(f"  unpacked {os.path.basename(zip_path)}  ({len(members)} entries)")

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)      # start fresh so old photos don't mix in
os.makedirs(DATA_DIR)

if SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    zips = sorted(z for z in glob.glob(os.path.join(DRIVE_FOLDER, "*.zip"))
                  if not os.path.basename(z).lower().startswith("noisy"))   # don't re-read our own output
    assert zips, f"No .zip files found in {DRIVE_FOLDER}. Make a folder 'SafeCrop' in My Drive and put your zips there."
elif SOURCE == "upload":
    from google.colab import files
    os.makedirs("/content/uploads", exist_ok=True)
    uploaded = files.upload()    # a "Choose files" button appears below; pick your zip file(s)
    zips = []
    for name, data in uploaded.items():
        path = os.path.join("/content/uploads", name)
        with open(path, "wb") as f:
            f.write(data)
        zips.append(path)
    zips = [z for z in zips if z.lower().endswith(".zip")]
    assert zips, "Please choose at least one .zip file."
else:
    raise ValueError('SOURCE must be "drive" or "upload"')

print("Zip files found:", [os.path.basename(z) for z in zips])
for z in zips:
    unpack(z)

## Settings

Change these if your folders are somewhere else. Everything else can stay as it is.

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def norm(name):
    # "Clean Images" -> "clean_images", "Rice stem borer(info rich)" -> "rice_stem_borer_info_rich"
    return "_".join("".join(ch if ch.isalnum() else " " for ch in name.lower()).split())

def has_photo_subfolders(path):
    # True if at least one subfolder of `path` holds images
    try:
        subs = [os.path.join(path, d) for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]
    except OSError:
        return False
    return any(f.lower().endswith(IMG_EXT) for s in subs for f in os.listdir(s))

def find_clean_dirs(top=DATA_DIR):
    # Every folder named clean / clean images / clean_images (any capitals) in ALL attached datasets.
    # If none has that name, every top-most folder whose subfolders contain photos.
    named, any_ = [], []
    for root, dirs, files in sorted(os.walk(top)):
        if has_photo_subfolders(root):
            any_.append(root)
            if norm(os.path.basename(root)) in ("clean", "clean_images", "cleanimages"):
                named.append(root)
    found = named or any_
    # drop folders that sit inside another found folder (e.g. a duplicate "clean images/clean images")
    found = [p for p in found if not any(p != q and p.startswith(q + os.sep) for q in found)]
    if not found:
        raise FileNotFoundError(f"No folder of disease subfolders with photos found under {top}. "
                                "Check that your zip has a 'clean images' folder with one subfolder per disease.")
    return found

# Where the clean photos are (found automatically in the unpacked zips) and where results go.
INPUT_DIRS = find_clean_dirs()
OUTPUT_DIR = "/content/noisy_images"
# You can also list the folders yourself, e.g.
# INPUT_DIRS = ["/content/data/clean/clean images", "/content/data/rice_stem_borer_clean/clean images"]

IMAGE_SIZE = 224          # every photo becomes 224 x 224, CLIP's input size and the benchmark's
PHOTOS_PER_DISEASE = None # None = use every photo in each folder; a number (e.g. 20) = a fixed random pick of that many
SEED = 299                # same seed = exactly the same noisy images every time you run

NOISE_TYPES = ["gaussian_noise", "motion_blur", "defocus_blur", "low_light", "jpeg_compression"]
LEVELS = {1: 1, 2: 3, 3: 5}   # our level -> library severity (low_light uses its own 1/2/3)

# True: one noise type per photo (640 images, matches the labelling plan).
# False: every photo gets all five types (2,080 images); fine for CLIP and the SVM, too many for Gemini's free tier.
ONE_TYPE_PER_PHOTO = True

# Folder name -> (display name, group, origin). Edit after the field trip when the local diseases change.
DISEASES = {
    "potato_late_blight":       ("Potato late blight",       "A (info-rich)",   "Foreign"),
    "apple_scab":               ("Apple scab",               "A (info-rich)",   "Foreign"),
    "rice_stem_borer":          ("Rice stem borer",          "A (info-rich)",   "Local"),
    "black_gram_yellow_mosaic": ("Black gram yellow mosaic", "A (info-rich)",   "Local"),
    "pear_leaf_spot":           ("Pear leaf spot",           "B (info-scarce)", "Foreign"),
    "grape_leaf_blight":        ("Grape leaf blight",        "B (info-scarce)", "Foreign"),
    "jute_mosaic":              ("Jute mosaic",              "B (info-scarce)", "Local"),
    "jute_dieback":             ("Jute dieback",             "B (info-scarce)", "Local"),
}

def match_disease(folder_name):
    # Match a folder to a DISEASES entry if every word of the entry appears in the folder name,
    # so "Rice stem borer(info rich)" -> rice_stem_borer and "Potato___Late_blight" -> potato_late_blight.
    words = set(norm(folder_name).split("_"))
    hits = [k for k in DISEASES if set(k.split("_")) <= words]
    return max(hits, key=lambda k: len(k.split("_"))) if hits else norm(folder_name)

import hashlib

# disease key -> list of photo paths, gathered from every input folder.
# If the same disease appears in two datasets, their photos are combined; exact duplicate files are kept once.
photos, seen, display_name = {}, set(), {}
print("Reading from:")
for base in INPUT_DIRS:
    print(" ", os.path.abspath(base))
    for d in sorted(os.listdir(base)):
        folder = os.path.join(base, d)
        if not os.path.isdir(folder):
            continue
        k = match_disease(d)
        display_name.setdefault(k, d)
        added = dups = 0
        # photos directly in the disease folder AND in any subfolders inside it
        for path in sorted(os.path.join(r, f) for r, _, fs in os.walk(folder) for f in fs):
            f = os.path.basename(path)
            if not f.lower().endswith(IMG_EXT) or f.startswith("._"):
                continue
            digest = hashlib.md5(open(path, "rb").read()).hexdigest()
            if digest in seen:
                dups += 1
                continue
            seen.add(digest)
            photos.setdefault(k, []).append(path)
            added += 1
        print(f"    {d!r:40} -> {k:26} {added} photos"
              + (f"  ({dups} skipped: already read from another folder)" if dups else "")
              + ("" if k in DISEASES else "   (not in DISEASES: group/origin blank)"))
assert photos, f"No disease folders with photos found in {INPUT_DIRS}"
folders = sorted(photos)   # disease keys
print("Total:", {k: len(v) for k, v in sorted(photos.items())})

## The building blocks

* `load_square` opens a photo, turns it the right way up (phones store rotation separately), cuts the central square and resizes it to 224 × 224. All photos must be the same size **before** noise is added, otherwise "a 10-pixel blur" means something different on a 4000-pixel photo than on a 256-pixel one.
* `low_light` is our own effect: only a small share of the light reaches the camera, so the picture gets darker and grainier (fewer "photons" means more random variation). The phone then brightens it part of the way back, and the colours fade.
* `make_noisy` applies one noise type at one level. It fixes the random seed first, so the result is identical every time.

In [ ]:
def load_square(path, size=IMAGE_SIZE):
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    w, h = img.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    img = img.crop((left, top, left + s, top + s)).resize((size, size), Image.BICUBIC)
    return np.array(img)


def low_light(img, level, photons=120):
    light = {1: 0.30, 2: 0.12, 3: 0.05}[level]          # share of light reaching the camera
    x = img.astype(np.float32) / 255.0
    signal = np.random.poisson(x * light * photons) / photons   # fewer photons -> grain
    out = signal * (1.0 / light) ** 0.5                  # the phone brightens it part of the way back
    grey = out.mean(axis=2, keepdims=True)
    out = grey + (out - grey) * {1: 0.90, 2: 0.75, 3: 0.60}[level]   # colours fade in the dark
    return (np.clip(out, 0, 1) * 255).astype(np.uint8)


def make_noisy(img, noise, level, seed):
    np.random.seed(seed)   # the library draws from NumPy's global random generator
    if noise == "low_light":
        out = low_light(img, level)
    else:
        out = corrupt(img, corruption_name=noise, severity=LEVELS[level])
    return np.clip(out, 0, 255).astype(np.uint8)


def seed_for(photo_id, noise, level):
    # a fixed number per photo, noise type and level, independent of file order
    return (zlib.crc32(f"{SEED}/{photo_id}/{noise}/{level}".encode()) & 0x7FFFFFFF)


NICE = {"gaussian_noise": "Sensor noise", "motion_blur": "Hand-shake blur", "defocus_blur": "Out of focus",
        "low_light": "Low light", "jpeg_compression": "Compression"}

## Look before you run

This shows the first photo of the first disease with every noise type at every level. Check two things:

1. **L1** should look only slightly worse than the clean photo, and **L3** should be hard to read.
2. **L2** should be borderline. If almost every L2 is still easy, or almost every L2 is hopeless, change `LEVELS` (for example `{1: 2, 2: 3, 3: 4}`) or the numbers inside `low_light`.

In [ ]:
first = folders[0]
first_file = os.path.basename(photos[first][0])
sample = load_square(photos[first][0])

fig, axes = plt.subplots(len(NOISE_TYPES), 4, figsize=(10, 2.6 * len(NOISE_TYPES)))
for r, noise in enumerate(NOISE_TYPES):
    for c in range(4):
        im = sample if c == 0 else make_noisy(sample, noise, c, seed_for("preview", noise, c))
        axes[r, c].imshow(im)
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title(["Clean (L0)", "L1 mild", "L2 medium", "L3 severe"][c])
    axes[r, 0].set_ylabel(NICE[noise], fontsize=11)
plt.suptitle(f"{DISEASES.get(first, (first,))[0]}: {first_file}", y=1.0)
plt.tight_layout()
plt.show()

## Make all the images

For each disease, the photos are shuffled with the fixed seed and handed the five noise types in turn, so every disease gets the same mix. Each image is saved as a **PNG** (saving as JPG would add compression damage to every image, even the clean ones), in folders by level:

```
noisy_images/L0/jute_mosaic/jute_mosaic_007_clean_L0.png
noisy_images/L2/jute_mosaic/jute_mosaic_007_motion_blur_L2.png
```

In [ ]:
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)          # start fresh so old files don't mix in
rows = []
for disease in folders:
    files = photos[disease]               # full paths
    if PHOTOS_PER_DISEASE is not None:
        if len(files) < PHOTOS_PER_DISEASE:
            print(f"Warning: {disease} has only {len(files)} photos (wanted {PHOTOS_PER_DISEASE})")
        elif len(files) > PHOTOS_PER_DISEASE:
            # a fixed random pick (not the first N by name, which could all be one kind of photo)
            keep = set(random.Random(f"{SEED}-pick-{disease}").sample(files, PHOTOS_PER_DISEASE))
            files = [p for p in files if p in keep]
    name, group, origin = DISEASES.get(disease, (display_name[disease], "", ""))

    order = list(range(len(files)))
    random.Random(f"{SEED}-{disease}").shuffle(order)
    assigned = {files[k]: NOISE_TYPES[i % len(NOISE_TYPES)] for i, k in enumerate(order)}

    for idx, fname in enumerate(files, start=1):
        photo_id = f"{disease}_{idx:03d}"
        clean = load_square(fname)
        noises = [assigned[fname]] if ONE_TYPE_PER_PHOTO else NOISE_TYPES
        versions = [(0, "clean", clean, assigned[fname])]
        for noise in noises:
            for level in (1, 2, 3):
                versions.append((level, noise, make_noisy(clean, noise, level, seed_for(photo_id, noise, level)), noise))
        for level, tag, arr, noise in versions:
            rel = os.path.join(f"L{level}", disease, f"{photo_id}_{tag}_L{level}.png")
            os.makedirs(os.path.dirname(os.path.join(OUTPUT_DIR, rel)), exist_ok=True)
            Image.fromarray(arr).save(os.path.join(OUTPUT_DIR, rel))
            rows.append({
                "image": rel, "photo_id": photo_id, "disease": disease, "disease_name": name,
                "group": group, "origin": origin, "noise": noise, "level": f"L{level}",
                "severity": 0 if level == 0 else (level if noise == "low_light" else LEVELS[level]),
                "source_file": os.path.basename(fname),
                "source_path": os.path.relpath(fname, DATA_DIR),
            })

meta = pd.DataFrame(rows)
meta.to_csv(os.path.join(OUTPUT_DIR, "metadata.csv"), index=False)
print(f"Saved {len(meta)} images and metadata.csv in {os.path.abspath(OUTPUT_DIR)}")

## Check the result

In the first table, each disease row should show the same number across L0 to L3 (its number of photos). The second shows how the noise types were shared out (for example 6 photos per type when a disease has 30 photos and there are 5 types).

In [ ]:
print("Images per disease and level:")
display(meta.pivot_table(index="disease", columns="level", values="image", aggfunc="count", fill_value=0))
print("Photos per disease and noise type (counted at L1):")
display(meta[meta.level == "L1"].groupby(["disease", "noise"]).size().unstack(fill_value=0))

per_photo = 1 + 3 * (1 if ONE_TYPE_PER_PHOTO else len(NOISE_TYPES))
missing = [p for p in meta.image if not os.path.exists(os.path.join(OUTPUT_DIR, p))]
bad = meta.groupby("photo_id").size().loc[lambda s: s != per_photo]
print("Missing files:", len(missing), "| photos with the wrong number of versions:", len(bad))

## A preview for the slides

One photo per noise type, from clean to severe, saved as `preview.png` next to the images. You can drop it straight into the Google Slides update.

In [ ]:
rng = random.Random(SEED)
fig, axes = plt.subplots(len(NOISE_TYPES), 4, figsize=(10, 2.6 * len(NOISE_TYPES)))
for r, noise in enumerate(NOISE_TYPES):
    pick = rng.choice(sorted(meta[(meta.noise == noise) & (meta.level == "L1")].photo_id.unique()))
    for c in range(4):
        row = meta[(meta.photo_id == pick) & (meta.level == f"L{c}") & ((meta.noise == noise) | (c == 0))].iloc[0]
        axes[r, c].imshow(Image.open(os.path.join(OUTPUT_DIR, row.image)))
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title(["Clean (L0)", "L1 mild", "L2 medium", "L3 severe"][c])
    axes[r, 0].set_ylabel(f"{NICE[noise]}\n{row.disease_name}", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "preview.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Pack everything into one zip, save a copy to Google Drive (if connected) and download it.
zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("Zip ready:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")

if os.path.isdir(DRIVE_FOLDER):
    out = os.path.join(DRIVE_FOLDER, "output")
    os.makedirs(out, exist_ok=True)
    for f in (zip_path, os.path.join(OUTPUT_DIR, "metadata.csv"), os.path.join(OUTPUT_DIR, "preview.png")):
        shutil.copy(f, out)
    print("Saved to Google Drive:", out)

from google.colab import files
files.download(zip_path)          # your browser downloads noisy_images.zip

## Next steps

1. **Look at `preview.png`** with the team. If L2 isn't borderline for most noise types, adjust `LEVELS` and run again; the seed keeps everything else identical.
2. **Keep `noisy_images.zip`** (it is also in Drive under `SafeCrop/output`). The inference notebook reads `metadata.csv` and runs the SVM, CLIP and Gemini on every image.
3. **Label "still recognizable?"** for the L1 to L3 images in Label Studio, comparing each one with its L0 partner (same `photo_id`).

## Mistakes to avoid

* **Don't save as JPG.** It adds compression damage to every image, including the clean ones.
* **Don't resize after adding noise.** Shrinking an image smooths the noise away; resize first, as this notebook does.
* **Keep all versions of a photo together** when you split data for training the SVM or fine-tuning CLIP. Split by `photo_id`, never by image; otherwise the model is tested on photos it has already seen.
* **Use original photos only,** not the "augmented" folders some datasets ship (rotated or flipped copies of the same leaf).

*Credits: noise functions from [imagecorruptions](https://github.com/bethgelab/imagecorruptions) (Michaelis et al., 2019), which reimplements the benchmark of Hendrycks & Dietterich, "Benchmarking Neural Network Robustness to Common Corruptions and Perturbations", ICLR 2019. The `low_light` effect is our own.*